# 04 — Eval + Plots

**Purpose:** Compare random baseline vs trained adapters over 30 episodes each. Generate 6 publication-ready PNGs in `plots/`.

**Runtime:** Colab T4 · ~1h

> Requires notebooks 02 + 03 to have completed (adapters on HF Hub).

In [ ]:
# Cell 1 — Install
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.13" peft accelerate bitsandbytes
!pip install -q "openenv-core[core] @ git+https://github.com/meta-pytorch/OpenEnv@v0.2.3"
!pip install -q wandb huggingface_hub matplotlib numpy
import sys; sys.path.insert(0, 'sdk_sovereign_pkg')
!mkdir -p plots
print('✓ Ready')

In [ ]:
# Cell 2 — Config
HF_USER  = 'ishansurdi'
ENV_URL  = 'https://ishansurdi-sdk-sovereign.hf.space'
N_EPISODES = 30

In [ ]:
# Cell 3 — Auth
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
print('✓ Auth OK')

In [ ]:
# Cell 4 — Load baseline model (fresh untrained adapters)
import server.llm_agents as la
baseline_model, baseline_tok = la.load_model_with_two_adapters()
baseline_agents = la.make_agent_pair(baseline_model, baseline_tok)
print('✓ Baseline model loaded (untrained adapters)')

In [ ]:
# Cell 5 — Load trained model with Hub adapters
trained_model, trained_tok = la.load_model_with_two_adapters()
trained_model.load_adapter(
    f'{HF_USER}/sdk-sovereign-lead-adapter', adapter_name='lead_adapter_trained'
)
trained_model.load_adapter(
    f'{HF_USER}/sdk-sovereign-auditor-adapter', adapter_name='auditor_adapter_trained'
)
trained_agents = la.make_agent_pair(trained_model, trained_tok)
trained_agents['lead'].adapter_name    = 'lead_adapter_trained'
trained_agents['auditor'].adapter_name = 'auditor_adapter_trained'
print('✓ Trained model loaded')

In [ ]:
# Cell 6 — Eval loop
import json
from pathlib import Path
from client import SDKSovereignEnv

def run_eval(agents, label, n=N_EPISODES):
    results = []
    for ep in range(n):
        with SDKSovereignEnv(base_url=ENV_URL).sync() as env:
            obs = env.reset()
            total = 0.0; transcript = []
            while not obs.done and obs.turn_index < 7:
                action = agents[obs.current_role](obs)
                transcript.append({'turn': obs.turn_index,
                                   'role': obs.current_role,
                                   'action_type': action.action_type})
                obs = env.step(action)
                total += obs.reward
            state = env.state()
            tr = state.test_results or {}
            results.append({
                'total_reward': total,
                'tests_passed': sum(tr.values()),
                'tests_total':  len(tr) or 3,
                'success': bool(tr and all(tr.values())),
                'turns':   state.step_count,
                'repo':    state.repo_id,
                'terminated': state.terminated_reason,
                'transcript': transcript,
            })
        if ep % 5 == 0:
            print(f'  [{label}] ep {ep}/{n}')
    return results

baseline_results = run_eval(baseline_agents, 'baseline')
trained_results  = run_eval(trained_agents,  'trained')
Path('eval_results.json').write_text(json.dumps(
    {'baseline': baseline_results, 'trained': trained_results}, indent=2
))
print('✓ Eval complete')

In [ ]:
# Cell 7 — Plot 1: pass rate bar chart
import matplotlib.pyplot as plt

b_rate = sum(r['success'] for r in baseline_results) / len(baseline_results)
t_rate = sum(r['success'] for r in trained_results)  / len(trained_results)

plt.figure(figsize=(6,4))
plt.bar(['Baseline (untrained)', 'Trained (two LoRAs)'], [b_rate, t_rate],
        color=['#bbbbbb', '#1f77b4'])
plt.ylabel('Pass rate (all tests passed)')
plt.title('SDK-Sovereign — pass rate, n=30 each')
plt.ylim(0, 1)
for i, v in enumerate([b_rate, t_rate]):
    plt.text(i, v + 0.02, f'{v:.0%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('plots/pass_rate_baseline_vs_trained.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Baseline: {b_rate:.0%}  |  Trained: {t_rate:.0%}')

In [ ]:
# Cell 8 — Plot 2: mean episode reward
import statistics

b_r = statistics.mean(r['total_reward'] for r in baseline_results)
t_r = statistics.mean(r['total_reward'] for r in trained_results)

plt.figure(figsize=(6,4))
plt.bar(['Baseline', 'Trained'], [b_r, t_r], color=['#bbbbbb', '#1f77b4'])
plt.ylabel('Mean episode reward')
plt.axhline(0, color='k', lw=0.5)
plt.title('Mean total reward per episode (n=30)')
plt.tight_layout()
plt.savefig('plots/mean_reward.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

In [ ]:
# Cell 9 — Plot 3 & 4: WandB reward curves
import wandb

api = wandb.Api()
for run_name in ['lead-grpo-round1', 'auditor-grpo-round1']:
    try:
        run = api.runs(f'{HF_USER}/sdk-sovereign', {'display_name': run_name})[0]
        h = run.history()
        plt.figure(figsize=(8,4))
        col = next((c for c in h.columns if 'reward' in c.lower()), None)
        if col:
            plt.plot(h['_step'], h[col], label=col)
        plt.xlabel('GRPO step'); plt.ylabel('Reward')
        plt.title(f'GRPO training — {run_name}')
        plt.legend(); plt.grid(alpha=0.3)
        role = run_name.split('-')[0]
        plt.savefig(f'plots/reward_curve_{role}.png', dpi=150, bbox_inches='tight')
        plt.show(); plt.close()
    except Exception as e:
        print(f'WandB run {run_name} not found: {e}')

In [ ]:
# Cell 10 — Plot 5: per-repo pass rate
import numpy as np
from collections import defaultdict

b_per = defaultdict(list); t_per = defaultdict(list)
for r in baseline_results: b_per[r['repo']].append(r['success'])
for r in trained_results:  t_per[r['repo']].append(r['success'])
repos = sorted(set(b_per) | set(t_per))
b_vals = [sum(b_per[r])/len(b_per[r]) if b_per[r] else 0 for r in repos]
t_vals = [sum(t_per[r])/len(t_per[r]) if t_per[r] else 0 for r in repos]

x = np.arange(len(repos)); w = 0.35
plt.figure(figsize=(8,4))
plt.bar(x - w/2, b_vals, w, label='Baseline', color='#bbbbbb')
plt.bar(x + w/2, t_vals, w, label='Trained',  color='#1f77b4')
plt.xticks(x, repos, rotation=15)
plt.ylabel('Pass rate'); plt.legend()
plt.title('Pass rate by repo')
plt.tight_layout()
plt.savefig('plots/per_repo_pass_rate.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

In [ ]:
# Cell 11 — Plot 6: turns to completion distribution
b_turns = [r['turns'] for r in baseline_results if r['success']]
t_turns = [r['turns'] for r in trained_results  if r['success']]

plt.figure(figsize=(7,4))
bins = list(range(1, 9))
plt.hist([b_turns, t_turns], bins=bins, label=['Baseline', 'Trained'],
         color=['#bbbbbb', '#1f77b4'], edgecolor='white')
plt.xlabel('Turns to completion (successful episodes only)')
plt.ylabel('Count'); plt.legend()
plt.title('Distribution of completion turns')
plt.tight_layout()
plt.savefig('plots/completion_turns.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

print('\n✅ All 6 plots saved to plots/')
import os; print('Files:', sorted(os.listdir('plots/')))